# Résoudre `6_shortage` avec OR-Tools
Sixième jeu de données : 15 000 modèles pour seulement **5 000 datasets**. Le commentaire de l'instance demande : *« Where are the files? »*. La demande de données vaut 28 fois l'offre, et l'énergie limite aussi (7,6 fois le plafond). **Tout est rare à la fois.**

On reprend le modèle à sources agrégées du notebook 14, avec l'énergie mise à l'échelle (notebooks 13 et 14). Ce notebook montre :
1. que les sources **doublent** le score : une cible dont tous les besoins sont couverts par des sources n'a besoin d'aucun dataset ;
2. comment **diagnostiquer** une perte : mesurer ce qui reste dans les réserves, puis le gaspillage ;
3. qu'une **marge** sur l'offre totale, calibrée sur ce gaspillage, récupère une bonne partie de la perte.

**Prérequis** : `datasets/6_shortage.json`, `test_solution.py`, `pandas`, `ortools`.

**Durée d'exécution** : environ 14 minutes (dont 12 pour le solveur).

## 1. Charger l'instance et le repère

In [1]:
from pathlib import Path
from collections import defaultdict, Counter
import sys, json, time, bisect, random
import pandas as pd
from ortools.linear_solver import pywraplp
pd.set_option("display.width", 150)

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))
from test_solution import getSolutionScore

DATASET = ROOT_DIR / "datasets" / "6_shortage.json"
DATASET_TXT = DATASET.read_text()
instance = json.loads(DATASET_TXT)
energy_cap = instance["energyCap"]
print("commentaire de l'instance :", instance["comment"])

VALUE  = {m["id"]: m["value"] for m in instance["models"]}
ENERGY = {m["id"]: m["energyCost"] for m in instance["models"]}
NEEDS  = {m["id"]: [(t, b["lowerBound"], b["upperBound"]) for t, b in m["dataRequirements"].items()] for m in instance["models"]}
SIZE   = {d["id"]: d["size"] for d in instance["datasets"]}
TYPE   = {d["id"]: d["dataType"] for d in instance["datasets"]}
COPY   = {d["id"]: d["isCopyrighted"] for d in instance["datasets"]}
IMPOSSIBLE = {m for m, needs in NEEDS.items() if any(lo > hi for _, lo, hi in needs)}
LOWER = {m: {t: lo for t, lo, hi in needs if lo <= hi} for m, needs in NEEDS.items()}
MONO = {m: needs[0][0] for m, needs in NEEDS.items() if len(needs) == 1 and m not in IMPOSSIBLE}
FREE_SUPPLY  = {t: sum(SIZE[d] for d in SIZE if TYPE[d] == t and COPY[d] == 0) for t in "ntic"}
TOTAL_SUPPLY = {t: sum(SIZE[d] for d in SIZE if TYPE[d] == t) for t in "ntic"}

WINNER = 669_798
print(f"{len(VALUE)} modèles, {len(SIZE)} datasets | gagnant du concours : {WINNER:,}")

commentaire de l'instance : Where are the files?
15000 modèles, 5000 datasets | gagnant du concours : 669,798


## 2. Ce qui change

In [2]:
print(f"énergie de tous les modèles : {sum(ENERGY.values()) / energy_cap:.2f} × le plafond")
print(f"datasets : {len(SIZE)} dont {sum(COPY.values()) / len(COPY):.0%} sous droits")
print(f"modèles à un seul besoin (sources possibles) : {len(MONO)} | modèles impossibles : {len(IMPOSSIBLE)}")
demand = {t: sum(LOWER[m].get(t, 0) for m in VALUE) for t in "ntic"}
mono_lower = pd.Series([LOWER[s][t] for s, t in MONO.items()])
print(f"besoin d'une source : médiane {mono_lower.median():,.0f}, 10 % des sources demandent moins de {mono_lower.quantile(0.1):,.0f}")
pd.DataFrame({"offre_totale": TOTAL_SUPPLY, "offre_libre": FREE_SUPPLY, "demande_si_tout": demand,
              "demande_sur_offre": {t: round(demand[t] / TOTAL_SUPPLY[t], 1) for t in "ntic"}})

énergie de tous les modèles : 7.57 × le plafond
datasets : 5000 dont 41% sous droits
modèles à un seul besoin (sources possibles) : 5841 | modèles impossibles : 272
besoin d'une source : médiane 29,157, 10 % des sources demandent moins de 5,935


,offre_totale,offre_libre,demande_si_tout,demande_sur_offre
n,6459411,3794759,176913867,27.4
t,6087321,3611587,174225750,28.6
i,6264424,3702287,176045094,28.1
c,6277125,3742205,179954866,28.7


**Lecture**
- La demande vaut **28 fois** l'offre. Chaque type offre environ 6 millions d'unités, de quoi remplir à peine 300 besoins.
- L'énergie limite aussi : il faut choisir environ un modèle sur huit.
- 5 841 modèles n'ont qu'un seul besoin, et certains ont un besoin minuscule. Une **source** couvre un besoin entier de la cible sans consommer de datasets pour ce besoin. Une cible dont **tous** les besoins sont couverts n'a besoin d'**aucun** dataset. Quand les données manquent à ce point, c'est le levier principal.

## 3. Un contrôle préalable
Le modèle agrégé suppose qu'un modèle à pleine valeur peut être servi en données libres. On vérifie, besoin par besoin, que c'est **possible**, sur des réserves pleines. Un modèle dont un besoin ne peut pas être rempli en données libres seules est **interdit à pleine valeur**.

In [3]:
class Pool:
    def __init__(self, ids):
        self.ids = sorted(ids, key=SIZE.get)
        self.sizes = [SIZE[d] for d in self.ids]
    def take(self, i):
        self.sizes.pop(i)
        return self.ids.pop(i)
    def put(self, d):
        i = bisect.bisect_left(self.sizes, SIZE[d])
        self.sizes.insert(i, SIZE[d]); self.ids.insert(i, d)
    def remove(self, d):
        i = bisect.bisect_left(self.sizes, SIZE[d])
        while self.ids[i] != d:
            i += 1
        self.take(i)

def find_pair(sizes, a, b):
    i, j, best = 0, bisect.bisect_right(sizes, b) - 1, None
    while i < j:
        s = sizes[i] + sizes[j]
        if s > b:
            j -= 1
        elif s < a:
            i += 1
        else:
            if best is None or s < best[0]:
                best = (s, i, j)
            j -= 1
    return best

def fill(pool, lower, upper):
    taken, total = [], 0
    while total < lower:
        a, b = lower - total, upper - total
        i = bisect.bisect_left(pool.sizes, a)
        if i < len(pool.sizes) and pool.sizes[i] <= b:
            total += pool.sizes[i]; taken.append(pool.take(i)); continue
        pair = find_pair(pool.sizes, a, b) if pool.sizes and a <= 2 * pool.sizes[-1] else None
        if pair:
            _, i, j = pair
            total += pool.sizes[i] + pool.sizes[j]
            taken.append(pool.take(j)); taken.append(pool.take(i)); continue
        j = bisect.bisect_right(pool.sizes, b) - 1
        if j < 0:
            break
        total += pool.sizes[j]; taken.append(pool.take(j))
    if lower <= total <= upper:
        return taken
    for d in taken:
        pool.put(d)
    return None


pools = {(t, c): Pool([d for d in SIZE if TYPE[d] == t and COPY[d] == c]) for t in "ntic" for c in (0, 1)}
NO_FULL = set()
for m, needs in NEEDS.items():
    for t, lo, hi in needs:
        if lo <= hi:
            got = fill(pools[(t, 0)], lo, hi)
            if got is None:
                NO_FULL.add(m)
            else:
                for d in got:
                    pools[(t, 0)].put(d)
print("modèles interdits à pleine valeur (un besoin impossible en données libres seules) :", sorted(NO_FULL))

modèles interdits à pleine valeur (un besoin impossible en données libres seules) : [255, 9946]


## 4. Le modèle
Celui du notebook 14, avec trois ajouts :
- l'**énergie en milliards** (ici la contrainte est active, il faut la garder, mais avec des coefficients raisonnables) ;
- l'interdiction de pleine valeur pour les modèles de la partie 3 ;
- un paramètre `total_margin` : la part de l'offre totale que le modèle a le droit d'utiliser (partie 7).

In [4]:
def solve_model(seconds, possible_sources=True, total_margin=1.0):
    t0 = time.time()
    S = pywraplp.Solver.CreateSolver("SCIP")
    x = {m: S.BoolVar(f"x{m}") for m in VALUE if m not in IMPOSSIBLE and m not in NO_FULL}   # pleine valeur
    y = {m: S.BoolVar(f"y{m}") for m in VALUE}                              # demi-valeur
    w = {s: S.BoolVar(f"w{s}") for s in MONO}                               # source
    c = {}                                                                  # besoin couvert par une source
    for m, needs in NEEDS.items():
        for t, lo, hi in needs:
            if lo > hi:
                c[(m, t)] = y[m]
            elif possible_sources:
                c[(m, t)] = S.BoolVar(f"c{m}_{t}")
                S.Add(c[(m, t)] <= y[m])
    for m in VALUE:
        S.Add((x[m] if m in x else 0) + y[m] + (w[m] if m in w else 0) <= 1)   # un seul rôle
    E = 1e9                                                                 # énergie en milliards
    S.Add(sum(ENERGY[m] / E * x[m] for m in x) + sum(ENERGY[m] / E * y[m] for m in y)
          + sum(ENERGY[s] / E * w[s] for s in w) <= energy_cap / E)
    for t in "ntic":
        S.Add(sum(var for (m, tt), var in c.items() if tt == t) == sum(w[s] for s in w if MONO[s] == t))
        S.Add(sum(LOWER[m][t] * x[m] for m in x if t in LOWER[m]) <= FREE_SUPPLY[t])
        S.Add(sum(LOWER[m][t] * x[m] for m in x if t in LOWER[m])
              + sum(LOWER[m][t] * (y[m] - c.get((m, t), 0)) for m in y if t in LOWER[m])
              + sum(LOWER[s][t] * w[s] for s in w if MONO[s] == t) <= TOTAL_SUPPLY[t] * total_margin)
    S.Maximize(sum(VALUE[m] * x[m] for m in x) + sum((VALUE[m] // 2) * y[m] for m in y))
    S.SetTimeLimit(seconds * 1000)
    params = pywraplp.MPSolverParameters()
    params.SetDoubleParam(params.RELATIVE_MIP_GAP, 0.0)
    status = S.Solve(params)
    val = lambda v: round(v.solution_value())
    sol = dict(full=[m for m in x if val(x[m])], half=[m for m in y if val(y[m])],
               sources=[s for s in w if val(w[s])], covered=[(m, t) for (m, t), var in c.items() if val(var)])
    obj, bound = S.Objective().Value(), S.Objective().BestBound()
    info = dict(statut="OPTIMAL" if status == pywraplp.Solver.OPTIMAL else "FEASIBLE", objectif=round(obj),
                borne=round(bound, 1), écart_prouvé=round(bound - obj, 1), secondes=round(time.time() - t0),
                pleins=len(sol["full"]), demi=len(sol["half"]), sources=len(sol["sources"]))
    used = sum(ENERGY[m] for m in sol["full"] + sol["half"] + sol["sources"])
    assert used <= energy_cap, "énergie exacte dépassée"
    return sol, info

solver_runs = {}

## 5. Les sources doublent le score

In [5]:
sol_old, solver_runs["sources : besoins impossibles seulement"] = solve_model(120, possible_sources=False)
sol, solver_runs["sources partout"] = solve_model(300)
pd.DataFrame(solver_runs).T

,statut,objectif,borne,écart_prouvé,secondes,pleins,demi,sources
sources : besoins impossibles seulement,FEASIBLE,351206,351275.9,69.9,121,1570,458,78
sources partout,FEASIBLE,670872,670947.0,75.0,302,992,1250,1317


**Lecture**
- Avec des sources seulement pour les besoins impossibles, on plafonne vers 351 000 points. Avec des sources partout, le solveur dépasse **670 000**, au-dessus du gagnant sur le papier. Le score **double**.
- Regardez les colonnes : plus d'un millier de sources, et le nombre de modèles à demi-valeur augmente fortement. Dans le plan final, les 1 239 cibles de sources n'utilisent **aucun dataset** : tous leurs besoins sont couverts par des sources. Les datasets servent aux modèles pleins et aux sources elles-mêmes.

## 6. Attribuer les datasets
`give`, `pack`, `State`, `complete` et `repair` viennent du notebook 14, avec une différence : ici l'énergie limite, donc `State` vérifie de nouveau le plafond, comme dans les notebooks 10 et 11.

Avec seulement 5 000 datasets, une attribution prend moins d'une seconde. On peut donc essayer **200 ordres** bruités.

In [6]:
def give(pools, model, needs, statuses):
    got = []
    for t, lo, hi in needs:
        res = None
        for s in statuses:
            res = fill(pools[(t, s)], lo, hi)
            if res is not None:
                break
        if res is None:
            for d in got:
                pools[(TYPE[d], COPY[d])].put(d)
            return None
        got += res
    return [[int(d), int(model)] for d in got]

def pack(sol, order_key):
    """order_key(modèle, besoins à remplir) -> clé de tri. Rend (plan, nombre de modèles ratés)."""
    pools = {(t, c): Pool([d for d in SIZE if TYPE[d] == t and COPY[d] == c]) for t in "ntic" for c in (0, 1)}
    covered = defaultdict(set)
    for m, t in sol["covered"]:
        covered[m].add(t)
    jobs = [(m, NEEDS[m], (0,), "plein") for m in sol["full"]]
    jobs += [(m, [n for n in NEEDS[m] if n[0] not in covered[m]], (1, 0), "demi") for m in sol["half"]]
    jobs += [(s, NEEDS[s], (1, 0), "source") for s in sol["sources"]]
    data, failed = {}, []
    for m, needs, statuses, kind in sorted(jobs, key=lambda j: order_key(j[0], j[1])):
        got = give(pools, m, needs, statuses)
        if got is None:
            failed.append((m, kind))
        else:
            data[m] = got
    for m, kind in failed:                                   # seconde chance à demi-valeur pour les pleins
        if kind == "plein":
            got = give(pools, m, NEEDS[m], (1, 0))
            if got is not None:
                data[m] = got
    # Appariement sources -> besoins couverts, type par type
    available = defaultdict(list)
    for s in sol["sources"]:
        if s in data:
            available[MONO[s]].append(s)
    model_mappings = []
    for m in sol["half"]:
        if m not in data:
            continue
        pairs = []
        for t in covered[m]:
            if available[t]:
                pairs.append((available[t].pop(), m))
        if len(pairs) == len(covered[m]):
            model_mappings += pairs
        else:                                                # cible incomplète : on la retire
            for s, _ in pairs:
                available[MONO[s]].append(s)
            data.pop(m)
    for t, unused in available.items():                      # sources sans cible : on les retire
        for s in unused:
            data.pop(s)
    plan = {"dataMappings": [p for got in data.values() for p in got],
            "modelMappings": [[int(s), int(m)] for s, m in model_mappings]}
    return plan, len(failed)

In [7]:
class State:
    """Solution modifiable, avec journal d'annulation. Sources et cibles sont gelées."""
    def __init__(self, plan):
        frozen_models = {x for pair in plan["modelMappings"] for x in pair}
        self.frozen_data = [p for p in plan["dataMappings"] if p[1] in frozen_models]
        self.frozen_sources = plan["modelMappings"]
        self.frozen = frozen_models
        self.data = {}
        for d, m in plan["dataMappings"]:
            if m not in frozen_models:
                self.data.setdefault(m, []).append(d)
        used = {d for d, _ in plan["dataMappings"]}
        self.pools = {(t, c): Pool([d for d in SIZE if d not in used and TYPE[d] == t and COPY[d] == c])
                      for t in "ntic" for c in (0, 1)}
        self.full = {m: not any(COPY[d] for d in ds) for m, ds in self.data.items()}
        self.energy = sum(ENERGY[m] for m in self.data) + sum(ENERGY[m] for m in frozen_models)
        self.log = []

    def remove(self, m):
        ds, full = self.data.pop(m), self.full.pop(m)
        for d in ds:
            self.pools[(TYPE[d], COPY[d])].put(d)
        self.energy -= ENERGY[m]
        self.log.append(("remove", m, ds, full))

    def add(self, m, statuses):
        if self.energy + ENERGY[m] > energy_cap:      # ici, l'énergie limite
            return False
        if statuses == (0,) and m in NO_FULL:
            return False
        got = give(self.pools, m, NEEDS[m], statuses)
        if got is None:
            return False
        self.data[m] = [d for d, _ in got]
        self.full[m] = not any(COPY[d] for d in self.data[m])
        self.energy += ENERGY[m]
        self.log.append(("add", m, None, None))
        return True

    def undo(self):
        while self.log:
            op, m, ds, full = self.log.pop()
            if op == "add":
                for d in self.data.pop(m):
                    self.pools[(TYPE[d], COPY[d])].put(d)
                self.full.pop(m); self.energy -= ENERGY[m]
            else:
                for d in ds:
                    self.pools[(TYPE[d], COPY[d])].remove(d)
                self.data[m], self.full[m] = ds, full
                self.energy += ENERGY[m]

    def plan(self):
        mappings = [[int(d), int(m)] for m, ds in self.data.items() for d in ds] + self.frozen_data
        return {"dataMappings": mappings, "modelMappings": self.frozen_sources}

TYPES_OF = {m: {t for t, _, _ in needs} for m, needs in NEEDS.items()}
gain_full = lambda m: VALUE[m] - VALUE[m] // 2

def complete(s):
    candidates = [m for m in VALUE if m not in s.data and m not in s.frozen and m not in IMPOSSIBLE]
    candidates.sort(key=lambda m: -VALUE[m] / max(1, sum(LOWER[m].values())))
    added = 0
    for m in candidates:
        s.log = []
        if s.add(m, (0,)) or s.add(m, (1, 0)):
            added += 1
    return added

def repair(s, max_partners=100_000):
    n_repairs, improved = 0, True
    while improved:
        improved = False
        for b in sorted([m for m in s.data if not s.full[m]], key=gain_full, reverse=True):
            if b not in s.data or s.full[b]:
                continue
            s.log = []; s.remove(b)
            if s.add(b, (0,)):
                n_repairs += 1; improved = True; continue
            s.undo()
            partners = sorted([a for a in s.data if s.full[a] and TYPES_OF[a] & TYPES_OF[b] and gain_full(a) < gain_full(b)],
                              key=gain_full)
            for a in partners[:max_partners]:
                s.log = []; s.remove(a); s.remove(b)
                if s.add(b, (0,)) and s.add(a, (1, 0)):
                    n_repairs += 1; improved = True; break
                s.undo()
    return n_repairs


def search(sol, n_orders=200, n_finish=5, seed=0):
    """Essaie les deux ordres de référence + n_orders ordres bruités, puis complète et répare les n_finish meilleurs."""
    orders = {"petits besoins": lambda m, needs: sum(lo for t, lo, hi in needs),
              "plus contraints": lambda m, needs: (min((hi for t, lo, hi in needs), default=0), sum(lo for t, lo, hi in needs))}
    rng = random.Random(seed)
    for k in range(n_orders):
        noise = {m: rng.uniform(0.7, 1.3) for m in VALUE}
        orders[f"bruité {k}"] = lambda m, needs, nz=noise: sum(lo for t, lo, hi in needs) * nz[m]
    packed = []
    for name, key in orders.items():
        plan, n_failed = pack(sol, key)
        packed.append((getSolutionScore(json.dumps(plan), DATASET_TXT, show_details=False)[0], name, n_failed, plan))
    packed.sort(key=lambda r: -r[0])
    rows, best = [], (0, None, None)
    for score0, name, n_failed, plan in packed[:n_finish]:
        state = State(plan)
        added = complete(state)
        n = repair(state)
        final = state.plan()
        score = getSolutionScore(json.dumps(final), DATASET_TXT, show_details=False)[0]
        rows.append(dict(ordre=name, ratés=n_failed, après_attribution=score0, ajouts=added, réparations=n, score=score))
        if score > best[0]:
            best = (score, name, final)
    return best, pd.DataFrame(rows).set_index("ordre")

t0 = time.time()
best_a, table_a = search(sol)
print(f"{time.time() - t0:.0f} s | sélection du solveur : {solver_runs['sources partout']['objectif']:,} | meilleur plan : {best_a[0]:,}")
table_a

21 s | sélection du solveur : 670,872 | meilleur plan : 667,879


,ratés,après_attribution,ajouts,réparations,score
ordre,,,,,
bruité 143,19,667198,12,5,667879
bruité 3,18,667144,12,3,667814
bruité 193,18,667144,11,1,667852
bruité 54,18,667140,13,4,667778
bruité 42,19,667133,13,6,667758


**Lecture** : on perd plusieurs milliers de points entre la sélection du solveur et le plan réel, même avec le meilleur des 200 ordres, la complétion et la réparation. Il faut comprendre pourquoi.

## 7. Diagnostiquer la perte
Première question : **quelle ressource manque ?** On regarde ce qui reste dans chaque réserve après la meilleure attribution brute.

In [8]:
plan_raw, _ = pack(sol, lambda m, needs: (min((hi for t, lo, hi in needs), default=0), sum(lo for t, lo, hi in needs)))
used = {(t, c): 0 for t in "ntic" for c in (0, 1)}
for d, m in plan_raw["dataMappings"]:
    used[(TYPE[d], COPY[d])] += SIZE[d]
left = pd.DataFrame({"libre_restant": {t: FREE_SUPPLY[t] - used[(t, 0)] for t in "ntic"},
                     "sous_droits_restant": {t: TOTAL_SUPPLY[t] - FREE_SUPPLY[t] - used[(t, 1)] for t in "ntic"}})
left["besoin_moyen"] = {t: round(sum(LOWER[m].get(t, 0) for m in sol["full"] + sol["half"]) /
                              max(1, sum(1 for m in sol["full"] + sol["half"] if t in LOWER[m]))) for t in "ntic"}
left

,libre_restant,sous_droits_restant,besoin_moyen
n,23777,22869,32811
t,16851,1996,33508
i,12581,1068,32273
c,25015,17119,33536


Les deux réserves sont **presque vides** : il reste à peine de quoi remplir un besoin par type. Ce n'est donc pas la répartition libre / sous droits qui coince, c'est l'**offre totale**.

Deuxième question : **où sont passées les données ?** Le modèle compte, pour chaque besoin, exactement sa borne basse. Mais `fill` remplit chaque besoin **un peu au-dessus**. Mesurons ce gaspillage.

In [9]:
use_by_need = Counter()
for d, m in best_a[2]["dataMappings"]:
    use_by_need[(m, TYPE[d])] += SIZE[d]
waste = Counter()
for (m, t), total in use_by_need.items():
    waste[t] += total - LOWER[m][t]
pd.DataFrame({"gaspillage": waste, "part_de_l_offre": {t: f"{waste[t] / TOTAL_SUPPLY[t]:.2%}" for t in "ntic"}})

,gaspillage,part_de_l_offre
i,53106,0.85%
t,42071,0.69%
n,50683,0.78%
c,45424,0.72%


**Lecture** : le gaspillage représente environ **0,8 % de l'offre** de chaque type, soit environ 50 000 unités, l'équivalent d'un à deux besoins moyens de modèle (environ 33 000). Le modèle, qui compte les bornes basses exactes, prévoit donc plus de données qu'il n'en existe réellement. Les modèles servis en dernier, souvent ceux qui ont les plus gros besoins, échouent à l'attribution, et chaque échec se répercute sur les modèles suivants.

Deux remèdes qui **ne marchent pas**, essayés pendant la préparation de ce notebook :
- une marge sur les **données libres** : elle fait moins bien, puisque ce n'est pas la ressource qui manque ;
- un `fill` qui choisit, pour finir, la combinaison qui gaspille le moins (dataset seul ou paire) : il fait **beaucoup moins bien** (environ 640 000 points). Les paires consomment de petits datasets qui manquent ensuite à d'autres modèles. Moins de gaspillage local, plus de blocages ailleurs.

Le remède qui marche : dire au modèle qu'il ne peut utiliser qu'environ **98,5 %** de l'offre totale, pour laisser la place du gaspillage.

In [10]:
sol_m, solver_runs["sources partout, marge 1,5 % sur l'offre totale"] = solve_model(300, total_margin=0.985)
display(pd.DataFrame(solver_runs).T)
t0 = time.time()
best_b, table_b = search(sol_m)
print(f"{time.time() - t0:.0f} s | meilleur plan : {best_b[0]:,}")
table_b

,statut,objectif,borne,écart_prouvé,secondes,pleins,demi,sources
sources : besoins impossibles seulement,FEASIBLE,351206,351275.9,69.9,121,1570,458,78
sources partout,FEASIBLE,670872,670947.0,75.0,302,992,1250,1317
"sources partout, marge 1,5 % sur l'offre totale",FEASIBLE,668749,668857.1,108.1,301,990,1242,1303


24 s | meilleur plan : 668,777


,ratés,après_attribution,ajouts,réparations,score
ordre,,,,,
plus contraints,4,667884,16,2,668777
bruité 156,14,667248,20,2,668322
bruité 40,13,667233,23,0,668278
bruité 19,15,667193,21,0,668323
bruité 110,16,667177,21,3,668207


**Lecture**
- Sur le papier, la sélection avec marge vaut environ 2 100 points de moins (668 749 contre 670 872). Mais une fois les datasets attribués, elle donne **900 points de plus** (668 777 contre 667 879). Avec l'ordre « plus contraints d'abord », seuls 4 modèles échouent, et après complétion et réparation on retrouve **la valeur de la sélection**.
- C'est le paradoxe déjà vu dans le notebook 13 : **une solution un peu moins bonne sur le papier peut être bien meilleure en réalité**, si elle est plus facile à réaliser.

## 8. Comparaison et vérification

In [11]:
best_score, best_name, best_plan = max(best_a, best_b, key=lambda b: b[0])
summary = pd.DataFrame({"score": [best_a[0], best_b[0], WINNER,
                                  solver_runs["sources partout"]["borne"],
                                  solver_runs["sources : besoins impossibles seulement"]["objectif"]]},
                       index=["plan sans marge", "plan avec marge 1,5 %", "gagnant du concours",
                              "borne du solveur (sans marge)", "sans sources sur besoins remplissables"])
summary["écart_au_gagnant"] = summary.score - WINNER
summary["part_du_gagnant"] = (summary.score / WINNER).map("{:.2%}".format)
display(summary)
score, valid, message = getSolutionScore(json.dumps(best_plan), DATASET_TXT, show_details=True)
print("score officiel =", score, "| valide =", valid, message)

,score,écart_au_gagnant,part_du_gagnant
plan sans marge,667879.0,-1919.0,99.71%
"plan avec marge 1,5 %",668777.0,-1021.0,99.85%
gagnant du concours,669798.0,0.0,100.00%
borne du solveur (sans marge),670947.0,1149.0,100.17%
sans sources sur besoins remplissables,351206.0,-318592.0,52.43%



Total value: 668777
Energy used: 998402986477706 / 998405027273048 (100.00%)
Number of used models: 3551 / 15000 (23.67%)
  Number of devalued models: 2560 (72.09%)
  Number of models used as data source: 1303 (36.69%)
Number of used datasets: 4989 / 5000 (99.78%)
score officiel = 668777 | valide = True 


## 9. Écrire le fichier de solution
On ne remplace la solution principale que si on l'améliore.

In [12]:
out_dir = ROOT_DIR / "solutions"; out_dir.mkdir(exist_ok=True)
out = out_dir / "6_shortage_main.json"
current = getSolutionScore(out.read_text(), DATASET_TXT, show_details=False)[0] if out.exists() else 0
if valid and score > current:
    out.write_text(json.dumps(best_plan))
    print(f"6_shortage_main.json mis à jour : {current:,} -> {score:,}")
else:
    print(f"6_shortage_main.json inchangé ({current:,})")

6_shortage_main.json mis à jour : 0 -> 668,777


## À retenir
1. **Quand tout manque, les sources deviennent l'essentiel.** Une cible entièrement couverte par des sources n'a besoin d'aucun dataset. Ici, le score double.
2. **Diagnostiquer avant de corriger.** Deux mesures simples, ce qui reste dans les réserves puis le gaspillage par type, ont désigné la vraie cause. Les deux remèdes « intuitifs » essayés avant ce diagnostic faisaient moins bien, voire beaucoup moins bien.
3. **Une marge calibrée sur une mesure.** On ne choisit pas une marge au hasard : 0,8 % de gaspillage mesuré a donné l'ordre de grandeur de la marge à réserver.
4. **Optimiser le modèle n'est pas optimiser la réalité.** La sélection la mieux notée par le solveur n'est pas celle qui donne le meilleur plan réel.
5. **Contrôler les hypothèses du modèle.** Le contrôle de la partie 3 écarte d'emblée les modèles qui ne peuvent pas être servis en données libres, au lieu de laisser le solveur les choisir à pleine valeur.